<a href="https://colab.research.google.com/github/armakoua-a11y/colab-git-Project2-AK/blob/main/Project2_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 2: Advanced Data Cleaning and Preparation

1. Load dataset

In [3]:
import pandas as pd

df = pd.read_csv('cleaned_stock_data.csv', index_col=0)
df.index = pd.to_datetime(df.index)

df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 4084889 entries, 1970-01-02 to 2018-08-24
Data columns (total 12 columns):
 #   Column     Dtype  
---  ------     -----  
 0   ticker     object 
 1   open       float64
 2   close      float64
 3   adj_close  float64
 4   low        float64
 5   high       float64
 6   volume     float64
 7   exchange   object 
 8   name       object 
 9   sector     object 
 10  industry   object 
 11  decade     object 
dtypes: float64(6), object(6)
memory usage: 405.1+ MB


2. Advanced Data Cleaning

In [4]:
# Handling Missing Values
# Imputing missing values using linear time interpolation
df['close'] = df['close'].interpolate(method='linear')
df['volume'] = df['volume'].fillna(method='ffill').fillna(method='bfill')


/tmp/ipykernel_3141/3311419919.py:4: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['volume'] = df['volume'].fillna(method='ffill').fillna(method='bfill')


In [6]:
# Outlier filtering and using IQR method
Q1 = df[['close', 'volume']].quantile(0.25)
Q3 = df[['close', 'volume']].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df['close'] = df['close'].clip(lower=lower_bound['close'], upper=upper_bound['close'])
df['volume'] = df['volume'].clip(lower=lower_bound['volume'], upper=upper_bound['volume'])

In [7]:
# Structural error corrections
df['close'] = df['close'].abs()
df['volume'] = df['volume'].apply(lambda x: max(0, x))


In [15]:
df.describe()

# Example checks
df[df['close'] < 0]
df[df['volume'] < 0]

# Fix
df = df[df['close'] > 0]
df = df[df['volume'] > 0]


3. Data Transformation

3.1 Feature Engineering

In [8]:
# Feature Engineering
df['rolling_mean_20'] = df['close'].rolling(window=20).mean()
df['rolling_volatility_20'] = df['close'].rolling(window=20).std()
df['daily_return'] = df['close'].pct_change()


3.2 Normalization / Standardization

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
numerical_cols = ['close', 'volume', 'rolling_mean_20', 'rolling_volatility_20']
df[numerical_cols] = scaler.fit_transform(df[numerical_cols].fillna(0))


3.3 Encoding Categorical Variables

In [10]:
# One-hot encoding of market categorical flags
df = pd.get_dummies(df, columns=['exchange'], drop_first=True)


4. Integration and Final Dataset

In [17]:
df.dropna(inplace=True)
print(df.head())


           ticker       open     close  adj_close        low       high  \
date                                                                      
1982-06-30    AIG  39.274403  0.999762  29.174292  39.274403  39.442963   
1982-07-12    AIG  37.420246  0.897836  27.796968  37.420246  37.757366   
1982-07-29    AIG  38.263046  0.934900  28.297829  38.094486  38.263046   
1982-09-21    AIG  51.410698  1.666913  38.189522  51.410698  51.579258   
1982-11-03    AIG  52.927738  1.666913  38.189522  51.242142  52.927738   

              volume                                name   sector  \
date                                                                
1982-06-30  0.029331  AMERICAN INTERNATIONAL GROUP, INC.  FINANCE   
1982-07-12  0.438122  AMERICAN INTERNATIONAL GROUP, INC.  FINANCE   
1982-07-29  0.378502  AMERICAN INTERNATIONAL GROUP, INC.  FINANCE   
1982-09-21  0.344106  AMERICAN INTERNATIONAL GROUP, INC.  FINANCE   
1982-11-03  0.905699  AMERICAN INTERNATIONAL GROUP, INC.  FI

5. Data Splitting

In [18]:
# Time Series Split (Important)
train = df[:'2015']
val = df['2016':'2018']
test = df['2019':]


In [11]:
# Split ratios: Train 70%, Val 15%, Test 15%
train_idx = int(len(df) * 0.7)
val_idx = int(len(df) * 0.85)

train_split = df.iloc[:train_idx]
val_split = df.iloc[train_idx:val_idx]
test_split = df.iloc[val_idx:]


In [20]:
from sklearn.model_selection import train_test_split

X = df.drop('close', axis=1)
y = df['close']

X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False)

6. Save Data

In [12]:
# Save out to analytical artifacts
train_split.to_csv('train_clean.csv', index=False)
val_split.to_csv('val_clean.csv', index=False)
test_split.to_csv('test_clean.csv', index=False)
